# Task termination

A deterministic FrozenLake task ends when the agent reaches the goal, or truncates after its episode budget. Terminal and reset observations are emitted on separate steps.


In [1]:
"""End a task on success, with an episode budget as a timeout."""

import gymnasium as gym

from repeat_task_gym import RepeatTaskEnv


def solved(*, reward, done, **kwargs):
    return done == 1 and reward > 0


def main():
    base = gym.make(
        "FrozenLake-v1",
        desc=["SF", "FG"],
        is_slippery=False,
        max_episode_steps=2,
    )
    env = RepeatTaskEnv(base, max_task_episodes=2, terminate_task=solved)
    try:
        env.reset(seed=0)
        env.step(0)  # Stay at the start.
        observation, _, terminated, truncated, _ = env.step(0)
        print("Inner timeout:", observation, terminated, truncated)
        assert observation["episode_done"] == 2
        assert not terminated and not truncated

        observation, reward, _, _, _ = env.step(0)  # Episode reset frame.
        assert observation["episode_done"] == 0 and reward == 0.0
        env.step(2)  # Right.
        observation, reward, terminated, truncated, _ = env.step(1)  # Down.
        print("Solved task:", observation, reward, terminated, truncated)
        assert terminated and not truncated  # Success wins over the budget.

        env.reset()
        for _ in range(2):
            env.step(0)
        env.step(0)  # Episode reset frame.
        env.step(0)
        observation, _, terminated, truncated, _ = env.step(0)
        print("Task timeout:", observation, terminated, truncated)
        assert not terminated and truncated
    finally:
        env.close()


if __name__ == "__main__":
    main()


Inner timeout: {'observation': 0, 'episode_done': 2} False False
Solved task: {'observation': 3, 'episode_done': 1} 1.0 True False
Task timeout: {'observation': 0, 'episode_done': 2} False True
